Connect to http://134.226.86.100:9090/tree with password `xilinx`.

In [1]:
! pwd
! hostname -I

/home/xilinx/jupyter_notebooks
134.226.86.100 192.168.2.99 


In [ ]:
from collections import deque

import numpy as np

from libs.tcp_cosim_utils import initialize_server


DATA_TYPE = "single"  # e.g. "int8", "uint8", "int16", "single"
BATCH_SIZE = 1        # Simulink Send/Receive frame width must match
OVERFLOW = "wrap"    # integer output: "wrap", "saturate", or "error"
SAMPLE_RATE = 4000.0  # Hz; matches the current Simulink Ts = 1/4000
FFT_SIZE = 4096       # 1.024 s window; frequency resolution 0.9765625 Hz
FFT_UPDATE_INTERVAL = 64
FFT_INPUT_PORT = 5002
FFT_OUTPUT_PORT = 5003


def process_y(x):
    return 2.0 * x


def make_fft_peak_processor(sample_rate, fft_size, update_interval):
    samples = deque(maxlen=fft_size)
    window = np.hanning(fft_size)
    latest_peak_hz = 0.0
    samples_since_fft = 0

    def process_fft_peak(x):
        nonlocal latest_peak_hz, samples_since_fft
        samples.append(float(x))
        samples_since_fft += 1
        if len(samples) == fft_size and samples_since_fft >= update_interval:
            values = np.asarray(samples) - np.mean(samples)
            magnitudes = np.abs(np.fft.rfft(values * window))
            magnitudes[0] = 0.0
            peak_bin = int(np.argmax(magnitudes))
            latest_peak_hz = peak_bin * sample_rate / fft_size
            samples_since_fft = 0
        return latest_peak_hz

    return process_fft_peak


process_fft_peak = make_fft_peak_processor(
    SAMPLE_RATE,
    FFT_SIZE,
    FFT_UPDATE_INTERVAL,
)

main_server = initialize_server(
    process_function=process_y,
    namespace=globals(),
    namespace_key="main_server",
    input_port=5000,
    output_port=5001,
    data_type=DATA_TYPE,
    batch_size=BATCH_SIZE,
    overflow=OVERFLOW,
)

fft_server = initialize_server(
    process_function=process_fft_peak,
    namespace=globals(),
    namespace_key="fft_server",
    input_port=FFT_INPUT_PORT,
    output_port=FFT_OUTPUT_PORT,
    data_type=DATA_TYPE,
    batch_size=BATCH_SIZE,
    overflow=OVERFLOW,
)


Output channel 'result' listening on port 5001
Input server listening on port 5000
Co-simulation server configured
Input: port=5000, type=single, batch=1, bytes=4
Output: port=5001, type=single
Output channel 'result' listening on port 5003
Input server listening on port 5002
Co-simulation server configured
Input: port=5002, type=single, batch=1, bytes=4
Output: port=5003, type=single


Simulink Send connected from ('134.226.169.95', 42730)
Simulink Receive [result] connected from ('134.226.169.95', 51144)
Simulink Receive [result] connected from ('134.226.169.95', 56336)
Simulink Send connected from ('134.226.169.95', 55450)
Simulink Send connected from ('134.226.169.95', 46538)
Simulink Send connected from ('134.226.169.95', 54574)
Simulink Receive [result] connected from ('134.226.169.95', 40034)
Simulink Receive [result] connected from ('134.226.169.95', 46710)
Simulink Send connected from ('134.226.169.95', 53378)
Simulink Send connected from ('134.226.169.95', 58644)
Simulink Receive [result] connected from ('134.226.169.95', 47978)
Simulink Receive [result] connected from ('134.226.169.95', 39980)
Simulink Send connected from ('134.226.169.95', 38232)
Simulink Receive [result] connected from ('134.226.169.95', 50706)
Simulink Send connected from ('134.226.169.95', 33748)
Simulink Receive [result] connected from ('134.226.169.95', 52630)
Simulink Send connected 